# Transport Equation — State Space Models with HiPPO Interpolation
## Reduced-Order Modelling via Structured State Spaces

This notebook applies three **state space model (SSM)** architectures to reconstruct the full transport (advection) field from sparse sensor measurements.
All models use the **HiPPO (High-order Polynomial Projection Operator)** framework to interpolate and compress sensor time-series history.

**Pipeline:**
1. Generate transport trajectories via analytical solver ($u(x,t) = u_0((x-ct) \bmod L)$)
2. Compute POD basis via truncated SVD
3. Extract sparse sensor measurements + sliding-window pairs
4. Train three SSM architectures mapping sensor windows → POD modes
5. Compare reconstruction quality

$$\frac{\partial u}{\partial t} + c\,\frac{\partial u}{\partial x} = 0 \qquad (\text{periodic advection, } c \in [0.3,\, 2.5])$$

### Models
| # | Model | Description |
|---|-------|-------------|
| 1 | **S4** (Structured State Space for Sequences) | Full HiPPO state matrix $A \in \mathbb{R}^{N \times N}$, discretised via bilinear transform |
| 2 | **S4D** (Diagonal State Space) | Diagonal approximation of HiPPO — efficient $O(N)$ recurrence |
| 3 | **S6 / Mamba** (Selective State Space) | Input-dependent $B$, $C$, $\Delta$ for adaptive selectivity |

## 1 · Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.extmath import randomized_svd
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Project paths
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'scripts'))
sys.path.insert(0, str(project_root / 'utils'))

from transport_data import generate_trajectories, compute_pod

# Reproducibility
np.random.seed(0)
torch.manual_seed(0)

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 11})

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports done ✓')

## 2 · Generate Transport Trajectories

The transport equation $u_t + c\,u_x = 0$ has the exact solution $u(x,t) = u_0((x - ct) \bmod L)$.
We generate an ensemble of trajectories with varying advection speeds $c \in [0.3, 2.5]$ and random initial conditions (Gaussian bumps, Fourier modes, and mixtures).

In [ ]:
# ── Simulation parameters ──────────────────────────────────────────
Nx       = 128
Nt       = 300
T        = 6.0
n_train  = 100
n_test   = 20
n_total  = n_train + n_test

L_domain = 2 * np.pi
x        = np.linspace(0, L_domain, Nx, endpoint=False)
t_grid   = np.linspace(0, T, Nt)
dt       = t_grid[1] - t_grid[0]

# Generate trajectories using the transport_data utility
all_snaps, all_ic, speeds = generate_trajectories(
    n_total=n_total, Nx=Nx, Nt=Nt, T=T,
    c_range=(0.3, 2.5), seed=42
)

snaps_train = all_snaps[:n_train]
snaps_test  = all_snaps[n_train:]

print(f'Domain : x ∈ [0, {L_domain:.2f}],  Nx={Nx}')
print(f'Time   : T={T},  Nt={Nt},  dt={dt:.4f}')
print(f'Trajectories: {n_train} train + {n_test} test = {n_total}')
print(f'Speed range: c ∈ [{speeds.min():.2f}, {speeds.max():.2f}]')
print(f'Snapshots shape: {all_snaps.shape}')

## 3 · Visualise Trajectories

In [ ]:
show_idx = [0, 3, 7, 15, 30, 60]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()
for k, si in enumerate(show_idx):
    vmax_i = np.abs(snaps_train[si]).max()
    im = axes[k].pcolormesh(x, t_grid, snaps_train[si], cmap='RdBu_r',
                             shading='auto', vmin=-vmax_i, vmax=vmax_i)
    axes[k].set_xlabel('x'); axes[k].set_ylabel('t')
    axes[k].set_title(f'Traj {si} (c={speeds[si]:.2f})', fontsize=10)
    plt.colorbar(im, ax=axes[k], fraction=0.03)
fig.suptitle('Space–time heatmaps — Transport training trajectories', fontsize=13)
plt.tight_layout(); plt.show()

## 4 · POD Basis & Sensor Setup

In [ ]:
# ── POD ────────────────────────────────────────────────────────────
Phi, S, S_full = compute_pod(snaps_train, n_svd=50, energy_threshold=0.99)
n_modes = Phi.shape[1]

energy = np.cumsum(S_full**2) / np.sum(S_full**2)
print(f'POD: {n_modes} modes capture {energy[n_modes-1]*100:.2f}% energy')

modes_train = snaps_train @ Phi   # (n_train, Nt, n_modes)
modes_test  = snaps_test  @ Phi

# ── Sensors ────────────────────────────────────────────────────────
n_sensors  = 6
sensor_pos = [int((i+1) * Nx / (n_sensors+1)) for i in range(n_sensors)]
sensor_x   = x[sensor_pos]
print(f'Sensors at x = {np.round(sensor_x, 2)}')

# ── Scaling ────────────────────────────────────────────────────────
scaler = StandardScaler()
A_train_flat = modes_train.reshape(-1, n_modes)
scaler.fit(A_train_flat)

A_train_sc = scaler.transform(modes_train.reshape(-1, n_modes)).reshape(n_train, Nt, n_modes)
A_test_sc  = scaler.transform(modes_test.reshape(-1, n_modes)).reshape(n_test, Nt, n_modes)

# ── Full-sequence dataset (no sliding windows) ─────────────────────
X_all = all_snaps[:, :, sensor_pos]  # (n_total, Nt, n_sensors)

# ── Train/val split BY TRAJECTORY ────────────────────────────────
val_frac = 0.15
n_val_traj = max(1, int(n_train * val_frac))
traj_perm  = np.random.permutation(n_train)
tr_idx = traj_perm[n_val_traj:]
vl_idx = traj_perm[:n_val_traj]

X_tr   = X_all[:n_train][tr_idx].astype(np.float32)
Y_tr   = A_train_sc[tr_idx].astype(np.float32)
X_vl   = X_all[:n_train][vl_idx].astype(np.float32)
Y_vl   = A_train_sc[vl_idx].astype(np.float32)
X_test = X_all[n_train:].astype(np.float32)
Y_test = A_test_sc.astype(np.float32)

print(f'Trajectories: {len(tr_idx)} train, {len(vl_idx)} val, {n_test} test')
print(f'Shapes:  X_tr {X_tr.shape}, Y_tr {Y_tr.shape}')
print(f'         X_test {X_test.shape}, Y_test {Y_test.shape}')

## 5 · HiPPO Framework

The **HiPPO (High-order Polynomial Projection Operator)** provides a mathematically principled way to compress a continuous signal history into a finite-dimensional state.

Given an input signal $f(t)$, HiPPO maintains a state $c(t) \in \mathbb{R}^N$ such that the optimal polynomial approximation of $f$ on $[0, t]$ is recovered as $f(\tau) \approx \sum_n c_n(t)\, P_n(\tau)$, where $P_n$ are (shifted/scaled) Legendre polynomials.

This leads to a linear ODE:
$$\dot{c}(t) = A\, c(t) + B\, f(t)$$

where the **HiPPO-LegS** matrix entries are:
$$A_{nk} = -\begin{cases}
(2n+1)^{1/2}(2k+1)^{1/2} & \text{if } n > k \\
n+1 & \text{if } n = k \\
0 & \text{if } n < k
\end{cases}, \qquad B_n = (2n+1)^{1/2}$$

For the **transport equation**, HiPPO is particularly well-suited because the solution is a pure translation — the polynomial projection efficiently captures shifted profiles through its Legendre basis.

In [ ]:
def make_hippo_legs(N):
    """Construct the HiPPO-LegS (Legendre Scaled) matrices A and B.
    
    A ∈ R^{N×N}, B ∈ R^{N×1}
    These define the continuous-time ODE: dc/dt = A c + B f(t)
    that optimally projects the input history onto Legendre polynomials.
    """
    A = np.zeros((N, N))
    B = np.zeros((N, 1))
    for n in range(N):
        B[n, 0] = (2*n + 1) ** 0.5
        for k in range(N):
            if n > k:
                A[n, k] = -(2*n + 1)**0.5 * (2*k + 1)**0.5
            elif n == k:
                A[n, k] = -(n + 1)
    return A, B


def discretize_bilinear(A, B, step):
    """Bilinear (Tustin) discretisation for numerical stability."""
    N = A.shape[0]
    I = np.eye(N)
    Ainv = np.linalg.solve(I - A * step / 2, I)
    Ad   = Ainv @ (I + A * step / 2)
    Bd   = (Ainv * step) @ B
    return Ad, Bd


# Verify
A_hippo, B_hippo = make_hippo_legs(64)
print(f'HiPPO A: {A_hippo.shape}, B: {B_hippo.shape}')
print(f'A is lower-triangular + diagonal: {np.allclose(np.triu(A_hippo, 1), 0)}')

---
## 6 · Model 1: S4 — Structured State Space for Sequences

**S4** ([Gu et al., 2022](https://arxiv.org/abs/2111.00396)) parameterises a linear state space model
$$x_{k+1} = \bar{A}\, x_k + \bar{B}\, u_k, \qquad y_k = C\, x_k + D\, u_k$$
where $\bar{A}, \bar{B}$ are the discretised HiPPO matrices.

The recurrence is computed **as a convolution** via the kernel $\bar{K}_k = C \bar{A}^k \bar{B}$:
$$y = \bar{K} * u$$

For the transport equation (a linear PDE), S4's time-invariant dynamics are a natural fit — the convolution kernel can capture the fixed advection pattern across all speeds when combined with POD.

In [ ]:
class S4Layer(nn.Module):
    """Single S4 layer: HiPPO-initialised SSM with convolutional kernel."""
    
    def __init__(self, d_input, d_state=64, d_output=None, dropout=0.1, max_len=512):
        super().__init__()
        d_output = d_output or d_input
        self.d_input  = d_input
        self.d_state  = d_state
        self.d_output = d_output
        
        # HiPPO initialisation + bilinear discretisation (computed in NumPy)
        A_np, B_np = make_hippo_legs(d_state)
        step_init = 0.01
        I_np = np.eye(d_state)
        Ainv = np.linalg.solve(I_np - A_np * step_init / 2, I_np)
        A_bar_np = Ainv @ (I_np + A_np * step_init / 2)
        B_bar_np = (Ainv * step_init) @ B_np
        
        self.register_buffer('A_bar', torch.tensor(A_bar_np, dtype=torch.float32))
        self.register_buffer('B_bar', torch.tensor(B_bar_np.squeeze(), dtype=torch.float32))
        
        # Precompute A_powers in NumPy float64 for numerical stability
        A_powers_np = np.zeros((max_len, d_state, d_state), dtype=np.float64)
        A_powers_np[0] = np.eye(d_state)
        for k in range(1, max_len):
            A_powers_np[k] = A_powers_np[k-1] @ A_bar_np.astype(np.float64)
        self.register_buffer('A_powers',
                             torch.tensor(A_powers_np, dtype=torch.float32))
        
        self.C = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.D = nn.Parameter(torch.randn(d_input) * 0.01)
        
        self.out_proj = nn.Linear(d_input, d_output)
        self.norm     = nn.LayerNorm(d_output)
        self.dropout  = nn.Dropout(dropout)
    
    def _compute_kernel(self, L):
        # A_powers[:L] shape (L, N, N), C shape (H, N), B_bar shape (N,)
        CB = torch.einsum('hn, lnm, m -> hl', self.C, self.A_powers[:L], self.B_bar)
        return CB  # (H, L)
    
    def forward(self, u):
        B, L, H = u.shape
        K = self._compute_kernel(L)
        u_f = torch.fft.rfft(u.transpose(1, 2), n=2*L)
        K_f = torch.fft.rfft(K, n=2*L)
        y_f = u_f * K_f.unsqueeze(0)
        y   = torch.fft.irfft(y_f, n=2*L)[..., :L]
        y = y + u.transpose(1, 2) * self.D.unsqueeze(0).unsqueeze(-1)
        y = y.transpose(1, 2)
        y = self.dropout(F.gelu(self.out_proj(y)))
        y = self.norm(y)
        return y


class S4Model(nn.Module):
    def __init__(self, n_sensors, n_modes, d_model=64, d_state=64,
                 n_layers=3, dropout=0.1, max_len=512):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, d_model)
        self.layers = nn.ModuleList([
            S4Layer(d_model, d_state=d_state, dropout=dropout, max_len=max_len)
            for _ in range(n_layers)
        ])
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.Tanh(),
            nn.Linear(128, n_modes),
        )
    
    def forward(self, x):
        h = self.input_proj(x)
        for layer in self.layers:
            h = h + layer(h)
        return self.decoder(h)   # seq-to-seq: (B, L, n_modes)


s4_model = S4Model(n_sensors, n_modes, d_model=64, d_state=64, n_layers=3, max_len=512).to(device)
n_params = sum(p.numel() for p in s4_model.parameters())
print(f'S4 model: {n_params:,} parameters')

---
## 7 · Model 2: S4D — Diagonal State Space Model

**S4D** ([Gu et al., 2022b](https://arxiv.org/abs/2206.11893)) simplifies S4 by restricting the state matrix to be **diagonal**:
$$A = V \Lambda V^{-1}, \qquad \Lambda = \text{diag}(\lambda_1, \dots, \lambda_N)$$

The recurrence becomes element-wise:
$$x_k[n] = \bar{\lambda}_n\, x_{k-1}[n] + \bar{B}_n\, u_k$$

For the transport equation, the eigenstructure of the HiPPO matrix already captures the essential frequency content. The diagonal form lets each frequency component evolve independently — matching the physics of linear advection where each Fourier mode phase-shifts at rate $ck$.

In [ ]:
class S4DLayer(nn.Module):
    """Diagonal SSM layer with HiPPO initialisation (S4D)."""
    
    def __init__(self, d_input, d_state=64, dropout=0.1):
        super().__init__()
        self.d_input = d_input
        self.d_state = d_state
        
        A_np, B_np = make_hippo_legs(d_state)
        eigs = np.linalg.eigvals(A_np)
        idx = np.argsort(eigs.real)
        Lambda = eigs[idx]
        
        self.log_neg_real = nn.Parameter(
            torch.log(-torch.tensor(Lambda.real, dtype=torch.float32).clamp(max=-1e-4))
        )
        self.imag = nn.Parameter(
            torch.tensor(Lambda.imag, dtype=torch.float32)
        )
        
        self.B_re = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.B_im = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.C_re = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.C_im = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        
        self.D = nn.Parameter(torch.randn(d_input) * 0.01)
        self.log_step = nn.Parameter(torch.zeros(d_input).uniform_(-4, -1))
        
        self.dropout = nn.Dropout(dropout)
        self.norm    = nn.LayerNorm(d_input)
    
    def _get_lambda(self):
        return -torch.exp(self.log_neg_real) + 1j * self.imag
    
    def forward(self, u):
        B, L, H = u.shape
        Lambda = self._get_lambda()
        step   = torch.exp(self.log_step).clamp(min=1e-5, max=1.0)
        avg_step   = step.mean()
        Lambda_bar = torch.exp(Lambda * avg_step)
        
        B_c = self.B_re + 1j * self.B_im
        C_c = self.C_re + 1j * self.C_im
        
        powers = Lambda_bar.unsqueeze(0).pow(
            torch.arange(L, device=u.device).unsqueeze(-1).float()
        )
        K = torch.einsum('hn,ln,hn->hl', C_c, powers, B_c * avg_step).real
        
        u_f = torch.fft.rfft(u.transpose(1, 2), n=2*L)
        K_f = torch.fft.rfft(K, n=2*L)
        y   = torch.fft.irfft(u_f * K_f.unsqueeze(0), n=2*L)[..., :L]
        
        y = y + u.transpose(1, 2) * self.D.unsqueeze(0).unsqueeze(-1)
        y = y.transpose(1, 2)
        y = self.dropout(F.gelu(y))
        y = self.norm(y)
        return y


class S4DModel(nn.Module):
    def __init__(self, n_sensors, n_modes, d_model=64, d_state=64,
                 n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, d_model)
        self.layers = nn.ModuleList([
            S4DLayer(d_model, d_state=d_state, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.Tanh(),
            nn.Linear(128, n_modes),
        )
    
    def forward(self, x):
        h = self.input_proj(x)
        for layer in self.layers:
            h = h + layer(h)
        return self.decoder(h)   # seq-to-seq: (B, L, n_modes)


s4d_model = S4DModel(n_sensors, n_modes, d_model=64, d_state=64, n_layers=3).to(device)
n_params = sum(p.numel() for p in s4d_model.parameters())
print(f'S4D model: {n_params:,} parameters')

---
## 8 · Model 3: S6 / Mamba — Selective State Space Model

**Mamba** ([Gu & Dao, 2023](https://arxiv.org/abs/2312.00752)) introduces **selectivity** — the SSM parameters $B$, $C$, and step size $\Delta$ are generated dynamically from the input:

$$B_k = \text{Linear}_B(u_k), \quad C_k = \text{Linear}_C(u_k), \quad \Delta_k = \text{softplus}(\text{Linear}_\Delta(u_k))$$

For the transport equation with **variable advection speed** $c$, selectivity is particularly valuable: the model can adapt its temporal dynamics to different speeds without needing $c$ as an explicit input. The input-dependent $\Delta_k$ effectively lets Mamba learn to adjust its "clock speed" to match the local advection rate.

In [ ]:
class MambaLayer(nn.Module):
    """Selective SSM layer (Mamba / S6) with standard diagonal A."""
    
    def __init__(self, d_model, d_state=64, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.d_model  = d_model
        self.d_state  = d_state
        d_inner       = d_model * expand
        self.d_inner  = d_inner
        
        self.in_proj = nn.Linear(d_model, 2 * d_inner, bias=False)
        self.conv1d  = nn.Conv1d(
            d_inner, d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=d_inner, bias=True
        )
        self.x_proj  = nn.Linear(d_inner, d_state * 2 + 1, bias=False)
        
        # Standard Mamba A initialisation (arange) — avoids NaN from HiPPO eigenvalues
        self.register_buffer(
            'A_log', torch.log(torch.arange(1, d_state + 1, dtype=torch.float32))
        )
        
        self.dt_proj = nn.Linear(1, d_inner, bias=True)
        with torch.no_grad():
            self.dt_proj.bias.uniform_(-4.0, -2.0)
        
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm     = nn.LayerNorm(d_model)
        self.dropout  = nn.Dropout(dropout)
    
    def forward(self, u):
        B, L, _ = u.shape
        xz = self.in_proj(u)
        x, z = xz.chunk(2, dim=-1)
        x = self.conv1d(x.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x = F.silu(x)
        
        x_proj = self.x_proj(x)
        B_inp  = x_proj[..., :self.d_state]
        C_inp  = x_proj[..., self.d_state:2*self.d_state]
        dt_raw = x_proj[..., -1:]
        dt = F.softplus(self.dt_proj(dt_raw))
        A = -torch.exp(self.A_log)
        
        y = self._scan(x, dt, A, B_inp, C_inp)
        y = y * F.silu(z)
        y = self.out_proj(y)
        y = self.dropout(y)
        y = self.norm(y)
        return y
    
    def _scan(self, x, dt, A, B, C):
        """Sequential scan — numerically stable."""
        batch, L, d_inner = x.shape
        N = A.shape[0]
        
        h = torch.zeros(batch, d_inner, N, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t, :].unsqueeze(-1)        # (B, d_inner, 1)
            B_t  = B[:, t, :].unsqueeze(1)           # (B, 1, N)
            C_t  = C[:, t, :].unsqueeze(1)           # (B, 1, N)
            x_t  = x[:, t, :].unsqueeze(-1)          # (B, d_inner, 1)
            
            A_bar = torch.exp(dt_t * A)              # (B, d_inner, N)
            B_bar = dt_t * B_t                       # (B, d_inner, N)
            
            h = A_bar * h + B_bar * x_t              # (B, d_inner, N)
            y_t = (h * C_t).sum(-1)                  # (B, d_inner)
            ys.append(y_t)
        
        return torch.stack(ys, dim=1)                # (B, L, d_inner)


class MambaModel(nn.Module):
    def __init__(self, n_sensors, n_modes, d_model=64, d_state=32,
                 n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, d_model)
        self.layers = nn.ModuleList([
            MambaLayer(d_model, d_state=d_state, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.Tanh(),
            nn.Linear(128, n_modes),
        )
    
    def forward(self, x):
        h = self.input_proj(x)
        for layer in self.layers:
            h = h + layer(h)
        return self.decoder(h)   # seq-to-seq: (B, L, n_modes)


mamba_model = MambaModel(n_sensors, n_modes, d_model=64, d_state=32, n_layers=3).to(device)
n_params = sum(p.numel() for p in mamba_model.parameters())
print(f'Mamba model: {n_params:,} parameters')

## 9 · Training Loop

In [ ]:
def train_model(model, X_tr, Y_tr, X_vl, Y_vl,
                epochs=200, batch_size=4, lr=1e-3, patience=30,
                label='model'):
    """Standard training loop with early stopping."""
    train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(Y_tr))
    val_ds   = TensorDataset(torch.tensor(X_vl), torch.tensor(Y_vl))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size)
    
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, patience=15, factor=0.5)
    
    best_val  = float('inf')
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    wait      = 0
    tr_losses, vl_losses = [], []
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        n_batches  = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            optimiser.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            epoch_loss += loss.item()
            n_batches  += 1
        tr_losses.append(epoch_loss / n_batches)
        
        model.eval()
        val_loss = 0.0
        n_vb = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_loss += F.mse_loss(pred, yb).item()
                n_vb += 1
        vl_losses.append(val_loss / n_vb)
        scheduler.step(vl_losses[-1])
        
        if vl_losses[-1] < best_val:
            best_val   = vl_losses[-1]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        
        if epoch % 20 == 0 or epoch == 1:
            print(f'  [{label}] Epoch {epoch:3d}  train={tr_losses[-1]:.4e}  '
                  f'val={vl_losses[-1]:.4e}  best={best_val:.4e}')
        
        if wait >= patience:
            print(f'  [{label}] Early stop at epoch {epoch}')
            break
    
    model.load_state_dict(best_state)
    model.to(device)
    return tr_losses, vl_losses

print('Training function ready ✓')

### 9.1 · Train S4

In [ ]:
print('Training S4...')
tr_s4, vl_s4 = train_model(s4_model, X_tr, Y_tr, X_vl, Y_vl,
                            epochs=200, lr=1e-3, patience=30, label='S4')

### 9.2 · Train S4D

In [ ]:
print('Training S4D...')
tr_s4d, vl_s4d = train_model(s4d_model, X_tr, Y_tr, X_vl, Y_vl,
                              epochs=200, lr=1e-3, patience=30, label='S4D')

### 9.3 · Train Mamba

In [ ]:
print('Training Mamba...')
tr_mamba, vl_mamba = train_model(mamba_model, X_tr, Y_tr, X_vl, Y_vl,
                                 epochs=200, lr=1e-3, patience=30, label='Mamba')

## 10 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (tr, vl, name) in zip(axes, [
    (tr_s4, vl_s4, 'S4'), (tr_s4d, vl_s4d, 'S4D'), (tr_mamba, vl_mamba, 'Mamba')
]):
    ax.semilogy(tr, label='train', alpha=0.8)
    ax.semilogy(vl, label='val',   alpha=0.8)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.set_title(name); ax.legend()
plt.suptitle('Training Curves — Transport SSM Models', fontsize=13)
plt.tight_layout(); plt.show()

## 11 · Evaluation & Comparison

In [ ]:
def predict(model, X):
    """Predict on full-sequence trajectories, batched."""
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(X, dtype=torch.float32).to(device)
        preds = []
        for i in range(0, len(Xt), 4):
            preds.append(model(Xt[i:i+4]).cpu().numpy())
    return np.concatenate(preds, axis=0)  # (n_traj, Nt, n_modes)


def compute_metrics(y_true, y_pred):
    """Metrics for 3D arrays (n_traj, Nt, n_modes)."""
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred)**2)
    num = np.sqrt(np.sum((y_true - y_pred)**2, axis=-1))
    den = np.sqrt(np.sum(y_true**2, axis=-1)) + 1e-12
    mre = np.mean(num / den)
    return {'mae': mae, 'mse': mse, 'mre': mre}


# Modal-level metrics on test set
results = {}
for name, model in [('S4', s4_model), ('S4D', s4d_model), ('Mamba', mamba_model)]:
    yp = predict(model, X_test)          # (n_test, Nt, n_modes)
    m  = compute_metrics(Y_test, yp)
    results[name] = m
    print(f'{name:6s}  MAE={m["mae"]:.4e}  MSE={m["mse"]:.4e}  MRE={m["mre"]*100:.2f}%')

# Field-level reconstruction
print('\n── Field-level MRE per test trajectory ──')
for name, model in [('S4', s4_model), ('S4D', s4d_model), ('Mamba', mamba_model)]:
    yp_sc = predict(model, X_test)       # (n_test, Nt, n_modes)
    yp_un = scaler.inverse_transform(yp_sc.reshape(-1, n_modes)).reshape(n_test, Nt, n_modes)
    yt_un = scaler.inverse_transform(Y_test.reshape(-1, n_modes)).reshape(n_test, Nt, n_modes)
    
    traj_mre = []
    for ti in range(n_test):
        fp = yp_un[ti] @ Phi.T   # (Nt, Nx)
        ft = yt_un[ti] @ Phi.T   # (Nt, Nx)
        err = np.sqrt(np.mean((fp - ft)**2)) / (np.sqrt(np.mean(ft**2)) + 1e-12)
        traj_mre.append(err)
    print(f'  {name:6s}  mean field MRE = {np.mean(traj_mre)*100:.2f}% '
          f'(±{np.std(traj_mre)*100:.2f}%)')

## 12 · Reconstruction Visualisation

In [ ]:
ti = 0  # test trajectory index

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle(f'Transport - Test trajectory {ti} (speed={speeds[n_train+ti]:.2f})', fontsize=14)

for row, (name, model) in enumerate([('S4', s4_model), ('S4D', s4d_model), ('Mamba', mamba_model)]):
    yp_sc  = predict(model, X_test[ti:ti+1])   # (1, Nt, n_modes)
    yp_un  = scaler.inverse_transform(yp_sc.reshape(-1, n_modes)).reshape(Nt, n_modes)
    field_pred = yp_un @ Phi.T                   # (Nt, Nx)
    field_true = snaps_test[ti]                   # (Nt, Nx)

    ax_t = axes[row, 0]
    ax_p = axes[row, 1]
    vmin, vmax = field_true.min(), field_true.max()

    im_t = ax_t.pcolormesh(x_grid, t_grid, field_true, shading='auto', vmin=vmin, vmax=vmax)
    ax_t.set_title(f'{name} - True'); ax_t.set_xlabel('x'); ax_t.set_ylabel('t')
    plt.colorbar(im_t, ax=ax_t)

    im_p = ax_p.pcolormesh(x_grid, t_grid, field_pred, shading='auto', vmin=vmin, vmax=vmax)
    ax_p.set_title(f'{name} - Predicted'); ax_p.set_xlabel('x'); ax_p.set_ylabel('t')
    plt.colorbar(im_p, ax=ax_p)

plt.tight_layout()
plt.show()

## Animation: true vs predicted field evolution with full sensor time series

In [ ]:
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 40   # MB

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

traj_idx = 0
frame_step = 2

# Ground truth & predictions
field_true = snaps_test[traj_idx]          # (Nt, Nx)
sensor_ts  = X_test[traj_idx]             # (Nt, n_sensors)

preds = {}
for name, model in [('S4', s4_model), ('S4D', s4d_model), ('Mamba', mamba_model)]:
    yp_sc = predict(model, X_test[traj_idx:traj_idx+1])  # (1, Nt, n_modes)
    yp_un = scaler.inverse_transform(yp_sc.reshape(-1, n_modes)).reshape(Nt, n_modes)
    preds[name] = yp_un @ Phi.T   # (Nt, Nx)

frames = list(range(0, Nt, frame_step))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Transport – trajectory {traj_idx}  (speed={speeds[n_train+traj_idx]:.2f})', fontsize=13)

# Top-left: field snapshot
ax_field = axes[0, 0]
ax_field.set_xlim(x_grid[0], x_grid[-1])
ax_field.set_ylim(field_true.min() - 0.1, field_true.max() + 0.1)
ax_field.set_xlabel('x'); ax_field.set_ylabel('u')
line_true, = ax_field.plot([], [], 'k-', lw=2, label='True')
lines_pred = {}
colors = {'S4': 'tab:blue', 'S4D': 'tab:orange', 'Mamba': 'tab:green'}
for nm, c in colors.items():
    lines_pred[nm], = ax_field.plot([], [], '--', color=c, lw=1.5, label=nm)
ax_field.legend(loc='upper right', fontsize=8)
time_txt = ax_field.set_title('')

# Top-right: sensor time series
ax_sen = axes[0, 1]
ax_sen.set_title('Sensor inputs')
ax_sen.set_xlabel('t'); ax_sen.set_ylabel('sensor value')
n_show = min(5, sensor_ts.shape[1])
for s in range(n_show):
    ax_sen.plot(t_grid, sensor_ts[:, s], lw=0.8, alpha=0.7, label=f's{s}')
vline = ax_sen.axvline(t_grid[0], color='red', ls='--', lw=1)
ax_sen.legend(fontsize=6, ncol=2, loc='upper right')

# Bottom: per-model error
for i, nm in enumerate(['S4', 'S4D']):
    axes[1, i].set_xlim(x_grid[0], x_grid[-1])
    axes[1, i].set_xlabel('x'); axes[1, i].set_ylabel('|error|')

err_lines = {}
for nm, c in colors.items():
    if nm == 'S4':
        err_lines[nm], = axes[1, 0].plot([], [], color=c, lw=1.2, label=nm)
    else:
        err_lines[nm], = axes[1, 1].plot([], [], color=c, lw=1.2, label=nm)
axes[1, 0].legend(fontsize=8); axes[1, 1].legend(fontsize=8)

def init():
    line_true.set_data([], [])
    for l in lines_pred.values(): l.set_data([], [])
    for l in err_lines.values(): l.set_data([], [])
    vline.set_xdata([t_grid[0]])
    return [line_true] + list(lines_pred.values()) + list(err_lines.values()) + [vline]

def update(frame):
    t = t_grid[frame]
    line_true.set_data(x_grid, field_true[frame])
    ax_field.set_title(f't = {t:.3f}')
    for nm in preds:
        lines_pred[nm].set_data(x_grid, preds[nm][frame])
        err = np.abs(field_true[frame] - preds[nm][frame])
        err_lines[nm].set_data(x_grid, err)
    # auto-scale error axes
    for ax in [axes[1, 0], axes[1, 1]]:
        ax.relim(); ax.autoscale_view()
    vline.set_xdata([t])
    return [line_true] + list(lines_pred.values()) + list(err_lines.values()) + [vline]

anim = FuncAnimation(fig, update, frames=frames, init_func=init,
                      blit=False, interval=80)
plt.close(fig)
HTML(anim.to_jshtml())

## 14 · Summary

| Model | Key idea | Computation | Strengths for Transport |
|-------|----------|-------------|------------------------|
| **S4** | Full HiPPO matrix, conv kernel via $\bar{A}^k$ | $O(LN^2)$ | Captures the exact shift structure of advection through Legendre polynomials |
| **S4D** | Diagonal HiPPO (eigenvalue decomposition) | $O(LN)$ | Each eigenmode evolves independently — natural fit for linear PDE where Fourier modes decouple |
| **Mamba** | Input-dependent $B$, $C$, $\Delta$ (selectivity) | $O(LN)$ | Adapts step size to different advection speeds $c$ without explicit conditioning |